# 🔐 CodeBERT — Vuln Detector v4 (function-level) — full-FT / **LoRA**

Train trên `detect_v4_functionlevel.jsonl` (3,558 hàm, cân bằng, đã khử confound + chống rò rỉ).
Bật/tắt LoRA bằng `USE_LORA` ở cell cấu hình (LoRA = như paper).

**Kaggle:** Add Data → upload jsonl · GPU · cài đặt → **Restart kernel** → Run All.

In [ ]:
# Cài bản ổn định cho Kaggle, rồi RESTART KERNEL
!pip install -q "transformers==4.46.3" "tokenizers==0.20.3" "huggingface_hub==0.25.2" "accelerate==1.0.1" "peft==0.13.2" scikit-learn
!pip uninstall -y torchao   # tránh lỗi peft<->torchao trên Kaggle (LoRA không cần torchao)
print("Xong -> Run -> Restart Kernel -> Run All")

## 1. Cấu hình  (đổi `USE_LORA` để so full-FT vs LoRA)

In [ ]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # 1 GPU cho ổn định (tránh scalar-gather warning trên 2xT4)
import json, glob, random, collections
import numpy as np, torch
import transformers
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, EarlyStoppingCallback, DataCollatorWithPadding)
from sklearn.metrics import (accuracy_score, f1_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

MODEL_NAME = "microsoft/codebert-base"
MAX_LEN    = 512
TRAIN_BS   = 16      # OOM? -> 8
EVAL_BS    = 32

# ── LoRA (như paper) ──
USE_LORA   = True                       # True = LoRA, False = full fine-tune
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.1
LORA_TARGETS = ["query", "key", "value"]   # attention của RoBERTa/CodeBERT

EPOCHS = 10 if USE_LORA else 8
LR     = 5e-4 if USE_LORA else 2e-5     # LoRA cần LR CAO hơn nhiều
WEIGHT_DECAY, WARMUP_RATIO, PATIENCE = 0.01, 0.06, 3
LABEL2ID = {"Safe": 0, "Vulnerable": 1}; ID2LABEL = {0: "Safe", 1: "Vulnerable"}
OUTPUT_DIR = "./codebert-v4-lora" if USE_LORA else "./codebert-v4-full"
SAVE_DIR   = OUTPUT_DIR + "-final"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("transformers", transformers.__version__, "| device:",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
      "| mode:", "LoRA" if USE_LORA else "full-FT", "| LR", LR)

## 2. Nạp dữ liệu (dùng sẵn split train/val/test)

In [ ]:
def find_data():
    for p in ["/kaggle/input/**/detect_v4_functionlevel.jsonl",
              "detect_v4_functionlevel.jsonl",
              "../DatasetBuild/output/detect_v4_functionlevel.jsonl",
              "DatasetBuild/output/detect_v4_functionlevel.jsonl"]:
        h = sorted(glob.glob(p, recursive=True))
        if h: return h[0]
    raise FileNotFoundError("Chưa thấy data -> Add Data: upload detect_v4_functionlevel.jsonl")

PATH = find_data(); print("DATA:", PATH)
rows = [json.loads(l) for l in open(PATH, encoding="utf-8") if l.strip()]
rows = [r for r in rows if r.get("code") and r.get("label") in LABEL2ID]

splits = {"train": [], "val": [], "test": []}
for r in rows: splits.get(r.get("split", "train"), splits["train"]).append(r)
for k, v in splits.items():
    nv = sum(x["label"] == "Vulnerable" for x in v)
    print(f"  {k:5s}: {len(v):5d} (Vuln {nv}/Safe {len(v)-nv}) | {dict(collections.Counter(x['source'] for x in v))}")

## 3. Tokenize (truncation 512)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class DS(torch.utils.data.Dataset):
    def __init__(self, items):
        self.enc = tokenizer([r["code"] for r in items], truncation=True, max_length=MAX_LEN)
        self.y   = [LABEL2ID[r["label"]] for r in items]
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        d = {k: self.enc[k][i] for k in self.enc}; d["labels"] = self.y[i]; return d

train_ds, val_ds, test_ds = DS(splits["train"]), DS(splits["val"]), DS(splits["test"])
collator = DataCollatorWithPadding(tokenizer)
ntr = sum(1 for r in splits["train"] if len(tokenizer(r["code"])["input_ids"]) > MAX_LEN)
print(f"Train bị cắt >512 token: {ntr}/{len(splits['train'])} ({100*ntr/len(splits['train']):.1f}%)")

## 4. Mô hình (+LoRA nếu bật) + Huấn luyện

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, id2label=ID2LABEL, label2id=LABEL2ID)

if USE_LORA:
    from peft import LoraConfig, get_peft_model, TaskType
    lora = LoraConfig(task_type=TaskType.SEQ_CLS, r=LORA_R, lora_alpha=LORA_ALPHA,
                      lora_dropout=LORA_DROPOUT, target_modules=LORA_TARGETS,
                      modules_to_save=["classifier"])   # head phân loại train đầy đủ
    model = get_peft_model(model, lora)
    model.print_trainable_parameters()
model = model.to(device)

def compute_metrics(p):
    pr = np.argmax(p.predictions, axis=-1); y = p.label_ids
    P, R, F, _ = precision_recall_fscore_support(y, pr, average="macro", zero_division=0)
    return {"accuracy": accuracy_score(y, pr), "precision": P, "recall": R,
            "f1_macro": F, "f1_vuln": f1_score(y, pr, pos_label=1, zero_division=0)}

args = TrainingArguments(
    output_dir=OUTPUT_DIR, num_train_epochs=EPOCHS,
    per_device_train_batch_size=TRAIN_BS, per_device_eval_batch_size=EVAL_BS,
    learning_rate=LR, weight_decay=WEIGHT_DECAY, warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine", fp16=torch.cuda.is_available(),
    eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
    metric_for_best_model="f1_macro", greater_is_better=True,
    logging_steps=50, save_total_limit=2, report_to="none", seed=SEED)

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                  data_collator=collator, tokenizer=tokenizer, compute_metrics=compute_metrics,
                  callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)])
trainer.train()

print("\nEpoch | val_acc | val_prec | val_recall | val_f1 | val_f1_vuln")
for h in trainer.state.log_history:
    if "eval_f1_macro" in h:
        print(f"  {round(h['epoch']):3d} | {h['eval_accuracy']:.4f} | {h['eval_precision']:.4f} | "
              f"{h['eval_recall']:.4f} | {h['eval_f1_macro']:.4f} | {h['eval_f1_vuln']:.4f}")

## 5. Đánh giá TEST — Accuracy / Precision / Recall / F1

In [ ]:
pred = trainer.predict(test_ds)
yp = np.argmax(pred.predictions, axis=-1); yt = pred.label_ids

print("=" * 58 + f"\nTEST (n={len(yt)}) — {'LoRA' if USE_LORA else 'full-FT'}\n" + "=" * 58)
print(classification_report(yt, yp, target_names=["Safe", "Vulnerable"], digits=4, zero_division=0))
print("Confusion [[TN FP][FN TP]]:\n", confusion_matrix(yt, yp))
P, R, F, _ = precision_recall_fscore_support(yt, yp, average="macro", zero_division=0)
print(f"\n>> Accuracy={accuracy_score(yt, yp):.4f}  Precision={P:.4f}  Recall={R:.4f}  F1={F:.4f}")

src = np.array([r["source"] for r in splits["test"]])
print("\n--- Theo nguồn ---")
print(f"{'source':10s} {'n':>4s} {'acc':>7s} {'prec':>7s} {'recall':>7s} {'f1':>7s}")
for s in sorted(set(src)):
    ix = np.where(src == s)[0]
    p, r, f, _ = precision_recall_fscore_support(yt[ix], yp[ix], average="macro", zero_division=0)
    print(f"{s:10s} {len(ix):4d} {accuracy_score(yt[ix], yp[ix]):7.3f} {p:7.3f} {r:7.3f} {f:7.3f}")

## 6. Lưu mô hình

In [ ]:
os.makedirs(SAVE_DIR, exist_ok=True)
trainer.save_model(SAVE_DIR); tokenizer.save_pretrained(SAVE_DIR)
print("Đã lưu:", SAVE_DIR, os.listdir(SAVE_DIR))

## 7. Thử inference

In [ ]:
model.eval()
@torch.no_grad()
def predict(code):
    enc = tokenizer(code, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(device)
    p = torch.softmax(model(**enc).logits, dim=-1)[0, 1].item()
    return ("Vulnerable" if p >= 0.5 else "Safe"), p

TEST_CODE = "function withdraw(uint amount) public {\n    require(balances[msg.sender] >= amount);\n    (bool ok,) = msg.sender.call{value: amount}(\"\");\n    require(ok);\n    balances[msg.sender] -= amount;\n}"
lbl, prob = predict(TEST_CODE)
print(f"Dự đoán: {lbl}  (P(Vulnerable)={prob:.3f})")